# Lesson 7.4: How do you tell the LLM to say "I don't know"?

**Companion notebook for Lesson 7.4**

---

| Section | What you will build |
|---|---|
| 1. The Problem | Simulate the hallucination failure mode |
| 2. Mini RAG Corpus | Product docs + sentence-transformer retriever |
| 3. Defense Layer 1 | Soft vs hard abstention instructions, side-by-side |
| 4. Defense Layer 2 | Two-step gated generation + confidence scoring |
| 5. Defense Layer 3 | Retrieval-aware abstention + threshold calibration |
| 6. FPAR Metric | Out-of-scope eval set + False-Positive Answer Rate |
| 7. Stacking Defenses | Complete pipeline combining all three layers |

**Required:** `sentence-transformers`, `numpy`, `matplotlib`  
**Optional (Section 8):** `anthropic` — enables real Claude API calls instead of mock LLM responses

In [ ]:
# Uncomment to install
# !pip install sentence-transformers numpy matplotlib
# !pip install anthropic   # optional — for real LLM calls in Section 8

In [ ]:
%matplotlib inline

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import numpy as np
import warnings
warnings.filterwarnings('ignore')

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size']      = 11
plt.rcParams['axes.grid']      = True
plt.rcParams['grid.alpha']     = 0.3

def show_plot():
    plt.tight_layout()
    plt.show()

print('Imports ready.')

---
## 1. The Problem: LLMs that Never Say "I Don't Know"

Three forces push models toward confident fabrication:

```
┌─────────────────────────────────────────────────────────────────────┐
│              WHY LLMS HALLUCINATE INSTEAD OF ABSTAINING             │
└─────────────────────────────────────────────────────────────────────┘

  1. Training bias       RLHF raters preferred confident, complete answers.
     toward helpfulness  "I don't know" got lower scores.
                         Model internalised: unhelpful = bad.

  2. The fluency reflex  LLMs predict the next likely token, not the true one.
                         "What's the warranty on X?" → "X comes with a Y-year
                         warranty…" is statistically likely even when X = fiction.

  3. Soft prompts get    "Answer based on the context" is a soft suggestion.
     soft compliance     The model uses context as inspiration, not constraint.
```

In RAG systems this becomes the **most insidious failure mode**: the model answers
questions your retriever found nothing for, pulling from parametric memory,
dressed up in your brand voice, served with confidence.

In [ ]:
# Illustrate the failure with a mock LLM that mirrors typical model behaviour

class MockHallucinatingLLM:
    """Simulates a model trained to always produce fluent answers."""

    def __init__(self):
        # Parametric memory: things the model "knows" from training data
        self._parametric_memory = {
            'quantum teleporter': 'Our quantum teleporter comes with a 2-year limited warranty '
                                  'covering manufacturing defects and entanglement failures.',
            'ai butler':          'The AI Butler module is configured via Settings > Integrations > AI. '
                                  'Enable the feature and paste your API key to activate.',
            'refund $10000':      'For orders over $10,000 we offer a 30-day satisfaction guarantee. '
                                  'Contact enterprise@company.com to initiate the process.',
        }

    def generate(self, question: str, context: str) -> str:
        q = question.lower()
        # When context is empty/irrelevant the model falls back to parametric memory
        for key, answer in self._parametric_memory.items():
            if any(word in q for word in key.split()):
                return answer
        return ('Based on our documentation, this feature follows standard industry practices. '
                'Please refer to the relevant section of our user guide for full details.')


llm_bad = MockHallucinatingLLM()

out_of_scope_questions = [
    ('What is the warranty on the Quantum Teleporter?', ''),
    ('How do I configure the AI Butler module?', ''),
    ('What is the refund policy for orders over $10,000?', ''),
]

print('=== Hallucinating LLM (no abstention guard) ===\n')
for q, ctx in out_of_scope_questions:
    answer = llm_bad.generate(q, ctx)
    print(f'Q: {q}')
    print(f'A: {answer}')
    print(f'⚠️  HALLUCINATION — none of this is in the docs')
    print()

---
## 2. Mini RAG Corpus: Product Documentation

We will use this corpus for all three defense layers.  
It is intentionally small so every retrieval result is visible.

In [ ]:
# Product documentation corpus — real questions have answers here;
# out-of-scope questions will have no matching documents.
CORPUS = [
    {'id': 'doc_0',
     'title': 'Starter Plan Overview',
     'text':  'The Starter plan includes 5GB storage, up to 3 team members, '
              'and email support. Suitable for freelancers and small projects. '
              'The Starter plan is billed monthly only and costs $9 per month.'},

    {'id': 'doc_1',
     'title': 'Pro Plan Limits and Quotas',
     'text':  'Pro plan limits: maximum file upload size is 100MB per file. '
              'Maximum total storage is 1TB. Maximum concurrent API connections: 50. '
              'Maximum team members: unlimited. Contact support to discuss Enterprise upgrades.'},

    {'id': 'doc_2',
     'title': 'Refund Policy',
     'text':  'We offer a 14-day money-back guarantee on all plans. '
              'To request a refund, contact billing@company.com within 14 days of purchase. '
              'Refunds are processed within 5-7 business days. Annual plan refunds are prorated.'},

    {'id': 'doc_3',
     'title': 'Two-Factor Authentication Setup',
     'text':  'Enable 2FA under Settings > Security > Two-Factor Authentication. '
              'We support authenticator apps (Google Authenticator, Authy) and SMS codes. '
              'Hardware keys (FIDO2/WebAuthn) are available on Pro and Enterprise plans only.'},

    {'id': 'doc_4',
     'title': 'API Rate Limits',
     'text':  'Free plan: 100 API requests per day. Starter plan: 1,000 per day. '
              'Pro plan: 50,000 per day. Enterprise: unlimited. '
              'Limits reset at midnight UTC. Temporary increases available on request.'},

    {'id': 'doc_5',
     'title': 'Data Export Options',
     'text':  'Export your data at any time from Settings > Data > Export. '
              'Supported formats: CSV, JSON, XLSX, PDF. '
              'Large exports are processed asynchronously and emailed when ready. '
              'Data export is available on all paid plans.'},

    {'id': 'doc_6',
     'title': 'Password Reset',
     'text':  'To reset your password, click "Forgot password" on the login page. '
              'Enter your email address and check your inbox for a reset link. '
              'Reset links expire after 1 hour. If you do not receive an email, check spam.'},

    {'id': 'doc_7',
     'title': 'Enterprise Plan Features',
     'text':  'Enterprise plan includes SSO/SAML, custom file upload limits up to 10GB, '
              'dedicated Slack support, SLA guarantees, audit logs, and custom data retention. '
              'Contact sales@company.com for Enterprise pricing and onboarding.'},
]

print(f'Corpus: {len(CORPUS)} documents')
for doc in CORPUS:
    print(f'  [{doc["id"]}] {doc["title"]}')

In [ ]:
import os
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '60'
os.environ['TOKENIZERS_PARALLELISM']   = 'false'

from sentence_transformers import SentenceTransformer, util

print('Loading all-MiniLM-L6-v2 (cached)...')
embedder = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')

# Pre-compute and store document embeddings
doc_texts  = [doc['text'] for doc in CORPUS]
doc_embeds = embedder.encode(doc_texts, convert_to_tensor=True, show_progress_bar=False)

print(f'Embedded {len(CORPUS)} documents (dim={doc_embeds.shape[1]})')


def retrieve(query: str, top_k: int = 3, return_scores: bool = False):
    """Return top_k documents ranked by cosine similarity."""
    q_emb   = embedder.encode(query, convert_to_tensor=True, show_progress_bar=False)
    scores  = util.cos_sim(q_emb, doc_embeds)[0].cpu().numpy()
    indices = np.argsort(scores)[::-1][:top_k]
    docs    = [CORPUS[i] for i in indices]
    if return_scores:
        return docs, [float(scores[i]) for i in indices]
    return docs


print('Retriever ready.')

In [ ]:
# Quick sanity check — in-scope vs out-of-scope
test_queries = [
    ('What is the Pro plan upload size limit?',   'in-scope'),
    ('How do I reset my password?',               'in-scope'),
    ('What is the warranty on the Quantum Teleporter?', 'out-of-scope'),
    ('How do I configure the AI Butler module?',  'out-of-scope'),
]

print(f'{"Query":<50} {"Type":<14} {"Top-1 score":<14} {"Top-1 doc"}')
print('-' * 100)
for q, qtype in test_queries:
    docs, scores = retrieve(q, top_k=1, return_scores=True)
    print(f'{q:<50} {qtype:<14} {scores[0]:.4f}         {docs[0]["title"]}')

Notice the pattern: in-scope questions return scores above 0.5; out-of-scope questions
still return *something* (the retriever always ranks your whole corpus), but with
scores below 0.35. This signal is the foundation of Defense Layer 3.

---
## 3. Defense Layer 1: The Explicit Abstention Instruction

The simplest fix — and the most underused one — is telling the model
**exactly what to say** when it cannot answer from the provided context.

In [ ]:
# Two prompt templates — soft (bad) vs hard (good)

SOFT_PROMPT = """Answer based on the context. If you don't know, say so.

Context:
{context}

Question: {question}

Answer:"""


HARD_PROMPT = """Answer ONLY using the provided context.
If the context does not contain information needed to answer the question,
respond with EXACTLY this sentence and nothing else:
"I don't have information about that in the provided documents."
Do not guess. Do not use prior knowledge. Do not provide partial answers
based on related information.

Context:
{context}

Question: {question}

Answer:"""


print('Soft prompt:')
print('-' * 60)
print(SOFT_PROMPT.split('\n')[0])
print()
print('Hard prompt differences:')
print('  1. "ONLY using" — hard constraint, not soft suggestion')
print('  2. "EXACTLY this sentence" — concrete escape phrase')
print('  3. "Do not guess. Do not use prior knowledge." — closes parametric-memory loophole')
print('  4. "Do not provide partial answers" — closes the related-but-not-actual loophole')

In [ ]:
# Mock LLM that simulates different behaviours for soft vs hard prompts.
# In Section 8 we replace this with real Claude API calls.

ABSTENTION_PHRASE = "I don't have information about that in the provided documents."


class MockLLM:
    """
    Simulates prompt-following behaviour:
      - soft prompts: ~70% chance of hallucinating when context is empty
      - hard prompts: abstains when context is empty, answers when it is relevant
    """

    HALLUCINATIONS = {
        'quantum teleporter': 'Our quantum teleporter comes with a 2-year limited warranty '
                              'covering manufacturing defects and entanglement failures.',
        'ai butler':          'The AI Butler module is configured via Settings > Integrations > AI.',
        'refund':             'For orders over $10,000 we offer a 30-day satisfaction guarantee.',
        'ceo':                'Our CEO is Jane Smith, who founded the company in 2018.',
        'lasagna':            'For a classic lasagna: layer bolognese, béchamel, and pasta sheets. '
                              'Bake at 180°C for 40 minutes.',
    }

    def generate(self, prompt: str) -> str:
        # Extract context and question from the prompt
        ctx_start = prompt.find('Context:') + len('Context:')
        ctx_end   = prompt.find('Question:')
        q_start   = prompt.find('Question:') + len('Question:')
        q_end     = prompt.find('Answer:')

        context  = prompt[ctx_start:ctx_end].strip()
        question = prompt[q_start:q_end].strip().lower()

        is_hard_prompt  = 'EXACTLY this sentence' in prompt
        has_context     = len(context) > 20

        if has_context:
            # Simulate extracting an answer from the context
            if 'upload' in question or 'file size' in question or 'limit' in question:
                return 'The maximum file upload size on the Pro plan is 100MB per file.'
            if 'password' in question or 'reset' in question:
                return 'Click "Forgot password" on the login page, then check your email for a reset link.'
            if 'refund' in question and '10,000' not in question:
                return 'We offer a 14-day money-back guarantee. Email billing@company.com to request it.'
            if '2fa' in question or 'two-factor' in question or 'authenticator' in question:
                return 'Enable 2FA under Settings > Security > Two-Factor Authentication.'
            return 'Based on the provided context: ' + context[:120] + '...'

        # No useful context — simulate soft vs hard prompt behaviour
        if is_hard_prompt:
            return ABSTENTION_PHRASE  # hard prompt → complies with abstention rule

        # Soft prompt → model drifts into parametric memory
        for key, hallucination in self.HALLUCINATIONS.items():
            if key in question:
                return hallucination  # confident fabrication
        return 'Based on general knowledge: this typically follows standard industry practices.'


mock_llm = MockLLM()
print('Mock LLM ready.')

In [ ]:
def rag_with_prompt(question: str, prompt_template: str, show_context: bool = False) -> str:
    docs = retrieve(question, top_k=3)
    context = '\n\n'.join(f'[{d["title"]}]\n{d["text"]}' for d in docs)

    prompt = prompt_template.format(context=context, question=question)
    answer = mock_llm.generate(prompt)

    if show_context:
        print(f'Context passed to LLM ({len(docs)} chunks):')
        for d in docs:
            print(f'  - {d["title"]}')
    return answer


# Side-by-side comparison on out-of-scope questions
oos_questions = [
    'What is the warranty on the Quantum Teleporter?',
    'How do I configure the AI Butler module?',
    "What's the refund policy for orders over $10,000?",
    'Who is the CEO of Apple?',
    'What is a good lasagna recipe?',
]

print(f'{"Question":<50} {"Soft prompt":<45} {"Hard prompt"}')
print('=' * 140)
for q in oos_questions:
    soft_ans = rag_with_prompt(q, SOFT_PROMPT)
    hard_ans = rag_with_prompt(q, HARD_PROMPT)
    soft_tag = '⚠️  HALLUCINATED' if soft_ans != ABSTENTION_PHRASE else '✅ abstained'
    hard_tag = '✅ abstained'     if hard_ans == ABSTENTION_PHRASE  else '⚠️  answered'
    q_short  = q[:47] + '...' if len(q) > 47 else q
    print(f'{q_short:<50} {soft_tag:<45} {hard_tag}')

In [ ]:
# Confirm hard prompt still answers in-scope questions correctly
in_scope_questions = [
    'What is the maximum file upload size on the Pro plan?',
    'How do I reset my password?',
    'Does the 14-day refund guarantee apply to annual plans?',
]

print('In-scope questions with hard prompt:\n')
for q in in_scope_questions:
    ans = rag_with_prompt(q, HARD_PROMPT)
    tag = '✅ answered' if ans != ABSTENTION_PHRASE else '⚠️  over-abstained'
    print(f'Q: {q}')
    print(f'A: {ans}')
    print(f'   {tag}\n')

---
## 4. Defense Layer 2: Confidence Scoring (Two-Step Generation)

Instead of asking the model to answer directly, first ask it to decide **whether it can answer**.
This is called **gated generation** or a **two-step generation** pattern.

```
┌─────────────────────────────────────────────────────────────────────┐
│                     TWO-STEP GENERATION PATTERN                     │
└─────────────────────────────────────────────────────────────────────┘

  Query + Context
       │
       ▼
  ┌──────────────────────────────────────────┐
  │  Step 1: Classification                  │
  │  "Can you answer this from the context?" │
  │  Output: YES / NO                        │
  └─────────────────┬────────────────────────┘
                    │
          ┌─────────┴──────────┐
         YES                  NO
          │                   │
          ▼                   ▼
  ┌────────────────┐   ┌──────────────────────────────┐
  │  Step 2:       │   │  Return abstention phrase    │
  │  Generation    │   │  (no LLM call needed)        │
  │  Answer the    │   └──────────────────────────────┘
  │  question      │
  └────────────────┘

  Why it works: classification is a multiple-choice task.
  Models are much better at picking YES/NO than at mid-sentence abstention.
```

In [ ]:
GATE_PROMPT = """You are an AI assistant. Your task is ONLY to decide whether
the provided context contains enough information to answer the question.

Respond with ONLY one of these two words:
  YES — if the context directly addresses the question
  NO  — if the context does not contain the answer

Do not answer the question. Do not explain your reasoning.

Context:
{context}

Question: {question}

Decision (YES or NO):"""


ANSWER_PROMPT = """Answer ONLY using the provided context. Be concise.

Context:
{context}

Question: {question}

Answer:"""


class MockGateLLM:
    """Simulates a model that correctly classifies YES/NO for gated generation."""

    def classify(self, question: str, context: str) -> str:
        q = question.lower()
        # If any doc title/content is clearly relevant, say YES
        relevant_keywords = [
            'upload', 'file size', 'password', 'reset', 'refund', 'money-back',
            'two-factor', '2fa', 'rate limit', 'api', 'export', 'enterprise',
            'starter plan', 'pro plan',
        ]
        if any(kw in q for kw in relevant_keywords) and len(context) > 100:
            return 'YES'
        # Clearly fictional / off-topic
        fictional = ['quantum teleporter', 'ai butler', 'ceo of apple', 'lasagna',
                     'world cup', '$10,000']
        if any(f in q for f in fictional):
            return 'NO'
        # Default: if context is short/empty, probably NO
        return 'YES' if len(context) > 150 else 'NO'

    def answer(self, question: str, context: str) -> str:
        return mock_llm.generate(
            ANSWER_PROMPT.format(context=context, question=question)
        )


gate_llm = MockGateLLM()


def gated_rag(question: str, verbose: bool = True) -> dict:
    docs    = retrieve(question, top_k=3)
    context = '\n\n'.join(f'[{d["title"]}]\n{d["text"]}' for d in docs)

    # Step 1: Gate
    decision = gate_llm.classify(question, context)

    if decision == 'NO':
        answer  = ABSTENTION_PHRASE
        gated   = True
    else:
        answer  = gate_llm.answer(question, context)
        gated   = False

    if verbose:
        label = '⛔ GATED — no LLM generation call' if gated else '✅ PASSED — LLM generated answer'
        print(f'Q: {question}')
        print(f'Gate decision: {decision}  →  {label}')
        print(f'A: {answer}\n')

    return {'answer': answer, 'gated': gated, 'gate_decision': decision}


print('=== Gated Generation: in-scope questions ===')
for q in in_scope_questions:
    gated_rag(q)

print('=== Gated Generation: out-of-scope questions ===')
for q in oos_questions[:3]:
    gated_rag(q)

### Variant: Ask for a Confidence Score

A confidence score (0-10) gives you a **machine-readable signal**
you can log, threshold, and alert on.
If confidence falls below 7 for a topic cluster, your retriever has a coverage gap —
you have turned a hidden failure into a measurable metric.

In [ ]:
CONFIDENCE_PROMPT = """Before answering, output a confidence score from 0 to 10
indicating how well the provided context supports your answer.

Format your response EXACTLY as:
CONFIDENCE: <0-10>
ANSWER: <your answer, or "I don't have information about that in the provided documents.">

Only provide an answer if confidence >= 7. Otherwise write the abstention phrase above.

Context:
{context}

Question: {question}"""


def parse_confidence_response(raw: str) -> dict:
    """Parse CONFIDENCE: N / ANSWER: ... format."""
    lines    = raw.strip().splitlines()
    conf_val = None
    answer   = ''
    for line in lines:
        if line.startswith('CONFIDENCE:'):
            try:
                conf_val = int(line.split(':', 1)[1].strip())
            except ValueError:
                conf_val = 0
        elif line.startswith('ANSWER:'):
            answer = line.split(':', 1)[1].strip()
    return {'confidence': conf_val, 'answer': answer, 'abstained': answer == ABSTENTION_PHRASE}


class MockConfidenceLLM:
    """Simulates a model that outputs CONFIDENCE + ANSWER."""

    SCORES = {
        'upload':            9, 'file size':         9, 'password':          9,
        'reset':             8, 'refund':             9, 'two-factor':        8,
        'rate limit':        8, 'export':             8, 'enterprise':        7,
        'quantum teleporter': 0, 'ai butler':         0, 'lasagna':           0,
        'ceo':               0, 'world cup':          0, '$10,000':           1,
    }

    def generate(self, prompt: str) -> str:
        ctx_start = prompt.find('Context:') + len('Context:')
        q_start   = prompt.find('Question:') + len('Question:')
        context   = prompt[ctx_start: prompt.find('Question:')].strip()
        question  = prompt[q_start:].strip().lower()

        conf = 3  # default low confidence for unknown topics
        for kw, score in self.SCORES.items():
            if kw in question:
                conf = score
                break

        if conf >= 7:
            # Generate an answer from context (reuse mock_llm logic)
            ans = mock_llm.generate(
                HARD_PROMPT.format(context=context, question=question)
            )
        else:
            ans = ABSTENTION_PHRASE

        return f'CONFIDENCE: {conf}\nANSWER: {ans}'


conf_llm = MockConfidenceLLM()

all_test_questions = in_scope_questions + oos_questions[:4]

results_conf = []
print(f'{"Question":<52} {"Confidence":<12} {"Abstained?":<12} {"Answer"}')
print('-' * 130)
for q in all_test_questions:
    docs    = retrieve(q, top_k=3)
    context = '\n\n'.join(f'[{d["title"]}]\n{d["text"]}' for d in docs)
    raw     = conf_llm.generate(CONFIDENCE_PROMPT.format(context=context, question=q))
    parsed  = parse_confidence_response(raw)
    results_conf.append({'question': q, **parsed})

    q_short  = q[:49] + '...' if len(q) > 49 else q
    ans_short = parsed['answer'][:50] + '...' if len(parsed['answer']) > 50 else parsed['answer']
    print(f'{q_short:<52} {parsed["confidence"]:<12} {str(parsed["abstained"]):<12} {ans_short}')

In [ ]:
# Visualise confidence scores across question types
in_q  = [r for r in results_conf if r['question'] in in_scope_questions]
oos_q = [r for r in results_conf if r['question'] in oos_questions]

fig, ax = plt.subplots(figsize=(13, 5))

y_in  = [r['confidence'] for r in in_q]
y_oos = [r['confidence'] for r in oos_q]
x_all = list(range(len(in_q) + len(oos_q)))

bars_in  = ax.bar(range(len(in_q)), y_in,
                  color='#2E7D32', alpha=0.8, label='In-scope questions')
bars_oos = ax.bar(range(len(in_q), len(in_q) + len(oos_q)), y_oos,
                  color='#E53935', alpha=0.8, label='Out-of-scope questions')

ax.axhline(7, color='#F57F17', linewidth=2, linestyle='--', label='Threshold = 7 (below → abstain)')
ax.axvspan(len(in_q) - 0.5, len(in_q) + len(oos_q) - 0.5,
           alpha=0.05, color='#E53935')

labels = [q[:30] + '...' for q in [r['question'] for r in in_q + oos_q]]
ax.set_xticks(x_all)
ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('Confidence Score (0-10)')
ax.set_ylim(0, 11)
ax.set_title('LLM Confidence Scores: In-scope vs Out-of-scope Questions', fontweight='bold')
ax.legend(fontsize=10)

for bar, r in zip(list(bars_in) + list(bars_oos), in_q + oos_q):
    tag = '✅' if not r['abstained'] else '🚫'
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
            f"{r['confidence']}\n{tag}", ha='center', fontsize=9)

show_plot()

---
## 5. Defense Layer 3: Retrieval-Aware Abstention

The cheapest, fastest, most reliable abstention is the one where **the LLM never
gets a chance to hallucinate**.

If your retriever returned nothing useful, why pay for an LLM call at all?

```
┌────────────────────────────────────────────────────────────────────┐
│                 RETRIEVAL-AWARE ABSTENTION PIPELINE                │
└────────────────────────────────────────────────────────────────────┘

  Query
    │
    ▼
  Retriever
    │
    ├─── top-chunk score < threshold ──▶  Return abstention phrase (no LLM call)
    │                                     ↑ cheap: 1 embed + 1 cosine
    │
    └─── top-chunk score ≥ threshold ──▶  LLM generation

  Analogy: a bouncer at the door.
  If the question doesn't have credible "ID" (decent retrieval score),
  it never gets to the LLM party.
```

In [ ]:
# Record top-1 retrieval scores for in-scope and out-of-scope questions

in_scope_labeled = [
    'What is the maximum file upload size on the Pro plan?',
    'How do I reset my password?',
    'What are the API rate limits for the Pro plan?',
    'Does the 14-day refund guarantee apply to annual plans?',
    'How do I enable two-factor authentication?',
    'What file formats can I export my data in?',
    'How many team members does the Starter plan allow?',
    'What support do Enterprise customers get?',
]

out_of_scope_labeled = [
    'What is the warranty on the Quantum Teleporter?',
    'How do I configure the AI Butler module?',
    "What's the refund policy for orders over $10,000?",
    'Who is the CEO of Apple?',
    'What is a good lasagna recipe?',
    'Who won the 2022 FIFA World Cup?',
    'How does the Flux Capacitor work?',
    'What is our policy on quantum entanglement refunds?',
]

in_scores  = []
oos_scores = []

for q in in_scope_labeled:
    _, scores = retrieve(q, top_k=1, return_scores=True)
    in_scores.append(scores[0])

for q in out_of_scope_labeled:
    _, scores = retrieve(q, top_k=1, return_scores=True)
    oos_scores.append(scores[0])

print(f'In-scope  top-1 scores:  min={min(in_scores):.3f}  max={max(in_scores):.3f}  mean={np.mean(in_scores):.3f}')
print(f'OOS       top-1 scores:  min={min(oos_scores):.3f}  max={max(oos_scores):.3f}  mean={np.mean(oos_scores):.3f}')

In [ ]:
# Visualise the score distributions to pick a threshold

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: distributions
bins = np.linspace(0, 1, 20)
axes[0].hist(in_scores,  bins=bins, alpha=0.7, color='#2E7D32', label='In-scope questions')
axes[0].hist(oos_scores, bins=bins, alpha=0.7, color='#E53935', label='Out-of-scope questions')
axes[0].axvline(0.40, color='#F57F17', linewidth=2, linestyle='--', label='Threshold = 0.40')
axes[0].set_xlabel('Top-1 Cosine Similarity Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Retrieval Score Distributions\nIn-scope vs Out-of-scope', fontweight='bold')
axes[0].legend()

# Right: precision/recall tradeoff across thresholds
thresholds = np.linspace(0.1, 0.9, 80)
fpr_list   = []  # false positive rate (OOS wrongly passed through)
fnr_list   = []  # false negative rate (in-scope wrongly abstained)

for t in thresholds:
    fp = sum(1 for s in oos_scores if s >= t) / len(oos_scores)
    fn = sum(1 for s in in_scores  if s <  t) / len(in_scores)
    fpr_list.append(fp)
    fnr_list.append(fn)

axes[1].plot(thresholds, fpr_list, color='#E53935', linewidth=2.5,
             label='OOS hallucination rate (want low)')
axes[1].plot(thresholds, fnr_list, color='#2E7D32', linewidth=2.5,
             label='In-scope over-abstention rate (want low)')
axes[1].axvline(0.40, color='#F57F17', linewidth=2, linestyle='--', label='Suggested threshold = 0.40')
axes[1].set_xlabel('Abstention Threshold')
axes[1].set_ylabel('Error Rate')
axes[1].set_title('Threshold Trade-off\nHallucination Rate vs Over-abstention Rate', fontweight='bold')
axes[1].legend(fontsize=9)

show_plot()

# Show where the two curves cross
crossover_idx = np.argmin(np.abs(np.array(fpr_list) - np.array(fnr_list)))
print(f'Curves cross at threshold ≈ {thresholds[crossover_idx]:.2f}')
print(f'At threshold 0.40:')
print(f'  OOS hallucination rate: {fpr_list[np.argmin(np.abs(thresholds - 0.40))]:.0%}')
print(f'  In-scope over-abstention: {fnr_list[np.argmin(np.abs(thresholds - 0.40))]:.0%}')

In [ ]:
# Production-ready answer_question() with retrieval-aware abstention

ABSTENTION_THRESHOLD = 0.40

def answer_question(query: str, threshold: float = ABSTENTION_THRESHOLD,
                    verbose: bool = True) -> dict:
    """
    RAG pipeline with retrieval-aware abstention.

    Returns a dict with: answer, sources, abstained, reason
    """
    docs, scores = retrieve(query, top_k=3, return_scores=True)

    # Retrieval-aware abstention check
    if not docs or scores[0] < threshold:
        result = {
            'answer':    ABSTENTION_PHRASE,
            'sources':   [],
            'abstained': True,
            'reason':    f'low_retrieval_confidence (top score={scores[0]:.3f} < {threshold})',
        }
    else:
        context = '\n\n'.join(f'[{d["title"]}]\n{d["text"]}' for d in docs)
        ans     = mock_llm.generate(HARD_PROMPT.format(context=context, question=query))
        result  = {
            'answer':    ans,
            'sources':   [d['title'] for d in docs],
            'abstained': False,
            'reason':    f'top_score={scores[0]:.3f}',
        }

    if verbose:
        status = '⛔ ABSTAINED' if result['abstained'] else '✅ ANSWERED'
        print(f'Q: {query}')
        print(f'   {status} | {result["reason"]}')
        print(f'   A: {result["answer"][:90]}...\n' if len(result['answer']) > 90
              else f'   A: {result["answer"]}\n')

    return result


print('=== Retrieval-Aware Abstention: in-scope ===')
for q in in_scope_questions:
    answer_question(q)

print('=== Retrieval-Aware Abstention: out-of-scope ===')
for q in oos_questions[:4]:
    answer_question(q)

---
## 6. FPAR: Testing Abstention with an Out-of-Scope Eval Set

Most RAG eval sets only test happy-path questions — ones with clear answers in the corpus.
That is how abstention bugs slip into production.

The metric you actually care about:

$$\text{FPAR} = \frac{\text{OOS questions where model gave a confident answer}}{\text{total OOS questions}}$$

**Target: FPAR < 5%.** Anything higher means your system is hallucinating in production right now.

### Four categories for your OOS eval set

| Category | Example |
|---|---|
| Adjacent topics | Questions about your industry but not your product |
| Made-up entities | Products, features, or people that don't exist |
| Off-topic entirely | "Who won the World Cup?", "Lasagna recipe?" |
| Plausible-but-absent | Things that *sound* like they could be in your docs but aren't |

In [ ]:
# OOS eval set — four categories
OOS_EVAL_SET = [
    # Adjacent topics
    {'q': 'What is the average SaaS refund rate industry benchmark?',   'category': 'adjacent'},
    {'q': 'How does OAuth 2.0 work in general?',                        'category': 'adjacent'},
    {'q': 'What is the standard file size limit for email attachments?', 'category': 'adjacent'},

    # Made-up entities
    {'q': 'What is the warranty on the Quantum Teleporter?',            'category': 'made-up'},
    {'q': 'How do I configure the AI Butler module?',                   'category': 'made-up'},
    {'q': 'What features does the Turbo Plan include?',                 'category': 'made-up'},
    {'q': 'How does the SmartSync feature work?',                       'category': 'made-up'},

    # Off-topic entirely
    {'q': 'Who won the 2022 FIFA World Cup?',                           'category': 'off-topic'},
    {'q': 'What is a good lasagna recipe?',                             'category': 'off-topic'},
    {'q': 'Who is the CEO of Apple?',                                   'category': 'off-topic'},

    # Plausible-but-absent
    {"q": "What's our refund policy for orders over $10,000?",          'category': 'plausible-absent'},
    {'q': 'Does the Pro plan include phone support?',                   'category': 'plausible-absent'},
    {'q': 'What happens to my data after I cancel?',                    'category': 'plausible-absent'},
    {'q': 'Is there a free trial for the Enterprise plan?',             'category': 'plausible-absent'},
]

print(f'OOS eval set: {len(OOS_EVAL_SET)} questions')
from collections import Counter
counts = Counter(item['category'] for item in OOS_EVAL_SET)
for cat, n in counts.items():
    print(f'  {cat:<22} {n}')

In [ ]:
def is_abstention(answer: str) -> bool:
    """Check whether the response is the designated abstention phrase."""
    abstention_markers = [
        "i don't have information",
        "i don't have enough information",
        "i'm unable to find",
        "not in the provided",
        "not available in",
        "i cannot find",
    ]
    a = answer.lower().strip()
    return any(marker in a for marker in abstention_markers)


def run_fpar_eval(rag_fn, eval_set: list, label: str = '') -> float:
    """Run the eval and return FPAR."""
    fabrications = []
    for item in eval_set:
        result = rag_fn(item['q'])
        answer = result['answer'] if isinstance(result, dict) else result
        if not is_abstention(answer):
            fabrications.append({'question': item['q'], 'category': item['category'],
                                  'answer': answer})

    fpar = len(fabrications) / len(eval_set)
    print(f'\n{"=" * 60}')
    print(f'FPAR Evaluation: {label}')
    print(f'{"=" * 60}')
    print(f'Total OOS questions : {len(eval_set)}')
    print(f'Fabricated answers  : {len(fabrications)}')
    print(f'FPAR                : {fpar:.1%}')

    if fabrications:
        print(f'\nFabricated answers (need fixing):')
        for f in fabrications:
            print(f'  ⚠️  [{f["category"]}] {f["question"]}')
            print(f'     → {f["answer"][:80]}...')
    else:
        print('\n✅ No fabricated answers — all OOS questions correctly abstained.')

    return fpar


# Wrapper for answer_question to match the expected interface
def rag_no_guard(q):
    """RAG with soft prompt — no real abstention."""
    docs    = retrieve(q, top_k=3)
    context = '\n\n'.join(f'[{d["title"]}]\n{d["text"]}' for d in docs)
    ans     = mock_llm.generate(SOFT_PROMPT.format(context=context, question=q))
    return {'answer': ans}


# Suppress per-question verbosity for the batch run
def rag_guarded_silent(q):
    return answer_question(q, verbose=False)


print('Running FPAR eval on both systems...')
fpar_unguarded = run_fpar_eval(rag_no_guard,        OOS_EVAL_SET, 'No abstention guard (soft prompt)')
fpar_guarded   = run_fpar_eval(rag_guarded_silent,  OOS_EVAL_SET, 'Defense Layer 3: retrieval-aware')

In [ ]:
# Per-category breakdown
categories = list(counts.keys())
fpar_by_cat_unguarded = {}
fpar_by_cat_guarded   = {}

for cat in categories:
    subset = [item for item in OOS_EVAL_SET if item['category'] == cat]
    fab_u  = sum(1 for item in subset if not is_abstention(rag_no_guard(item['q'])['answer']))
    fab_g  = sum(1 for item in subset if not is_abstention(rag_guarded_silent(item['q'])['answer']))
    fpar_by_cat_unguarded[cat] = fab_u / len(subset)
    fpar_by_cat_guarded[cat]   = fab_g / len(subset)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(categories))
w = 0.35
ax.bar(x - w/2, [fpar_by_cat_unguarded[c] for c in categories],
       w, color='#E53935', alpha=0.8, label='No guard (soft prompt)')
ax.bar(x + w/2, [fpar_by_cat_guarded[c] for c in categories],
       w, color='#2E7D32', alpha=0.8, label='Defense Layer 3')
ax.axhline(0.05, color='#F57F17', linewidth=2, linestyle='--', label='Target FPAR = 5%')

ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=11)
ax.set_ylabel('FPAR (False-Positive Answer Rate)')
ax.set_title('FPAR by OOS Question Category', fontweight='bold')
ax.set_ylim(0, 1.15)
ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(xmax=1))
ax.legend()

for i, cat in enumerate(categories):
    ax.text(i - w/2, fpar_by_cat_unguarded[cat] + 0.03,
            f"{fpar_by_cat_unguarded[cat]:.0%}", ha='center', fontsize=10)
    ax.text(i + w/2, fpar_by_cat_guarded[cat] + 0.03,
            f"{fpar_by_cat_guarded[cat]:.0%}", ha='center', fontsize=10)

show_plot()

print(f'\nOverall FPAR: {fpar_unguarded:.1%} (no guard) → {fpar_guarded:.1%} (with guard)')

---
## 7. Stacking All Three Defense Layers

The strongest abstention strategy uses all three layers together:

```
┌─────────────────────────────────────────────────────────────────────────┐
│                      STACKED ABSTENTION PIPELINE                        │
└─────────────────────────────────────────────────────────────────────────┘

  Query
    │
    ▼
  ┌──────────────────────────────────────────────────────┐
  │  Layer 3: Retrieval-Aware Check                      │
  │  top-chunk score < threshold?                        │
  └───────────────────┬──────────────────────────────────┘
              YES ────┤  →  abstain immediately (cheapest)
                      │ NO
                      ▼
  ┌──────────────────────────────────────────────────────┐
  │  Layer 2: Confidence Gate                            │
  │  Ask model: can you answer from this context?        │
  └───────────────────┬──────────────────────────────────┘
              NO ─────┤  →  abstain (before full generation)
                      │ YES
                      ▼
  ┌──────────────────────────────────────────────────────┐
  │  Layer 1: Hard Prompt Constraint                     │
  │  Generate answer with explicit abstention rule       │
  └──────────────────────────────────────────────────────┘

  Each layer catches what the others miss:
  - Layer 3 catches obvious OOS cheaply (no LLM)
  - Layer 2 catches borderline cases (relevant-but-insufficient context)
  - Layer 1 catches residual drift into parametric memory
```

In [ ]:
def stacked_rag(
    query: str,
    threshold: float = ABSTENTION_THRESHOLD,
    confidence_min: int = 7,
    verbose: bool = True,
) -> dict:
    """
    Complete RAG pipeline with all three abstention layers stacked.

    Layer 3 → Layer 2 → Layer 1 (each layer is a gate; early exit is cheap).
    """
    docs, scores = retrieve(query, top_k=3, return_scores=True)
    top_score = scores[0] if scores else 0.0

    # ── Layer 3: Retrieval-aware ─────────────────────────────────────────────
    if top_score < threshold:
        result = {
            'answer':    ABSTENTION_PHRASE,
            'abstained': True,
            'layer':     'Layer 3 — low retrieval score',
            'score':     top_score,
            'llm_calls': 0,
        }
        if verbose:
            print(f'⛔ [L3] {query[:60]}')
            print(f'       score={top_score:.3f} < {threshold} → abstained without LLM call\n')
        return result

    context = '\n\n'.join(f'[{d["title"]}]\n{d["text"]}' for d in docs)

    # ── Layer 2: Confidence gate ─────────────────────────────────────────────
    raw_conf = conf_llm.generate(
        CONFIDENCE_PROMPT.format(context=context, question=query)
    )
    parsed   = parse_confidence_response(raw_conf)
    conf_val = parsed['confidence'] or 0

    if conf_val < confidence_min:
        result = {
            'answer':    ABSTENTION_PHRASE,
            'abstained': True,
            'layer':     f'Layer 2 — low confidence ({conf_val}<{confidence_min})',
            'score':     top_score,
            'llm_calls': 1,
        }
        if verbose:
            print(f'⛔ [L2] {query[:60]}')
            print(f'       score={top_score:.3f} ✓  confidence={conf_val} < {confidence_min} → abstained\n')
        return result

    # ── Layer 1: Hard prompt with explicit abstention rule ───────────────────
    answer = mock_llm.generate(HARD_PROMPT.format(context=context, question=query))
    abstained = is_abstention(answer)

    result = {
        'answer':    answer,
        'abstained': abstained,
        'layer':     'Layer 1 — hard prompt' + (' (abstained)' if abstained else ''),
        'score':     top_score,
        'llm_calls': 2,
    }
    if verbose:
        if abstained:
            print(f'⛔ [L1] {query[:60]}')
            print(f'       All layers passed but hard prompt returned abstention\n')
        else:
            print(f'✅ [answered] {query[:60]}')
            print(f'   score={top_score:.3f}  confidence={conf_val}  answer: {answer[:80]}\n')
    return result


print('=== Stacked pipeline: in-scope questions ===')
for q in in_scope_questions:
    stacked_rag(q)

print('=== Stacked pipeline: out-of-scope questions ===')
for q in oos_questions:
    stacked_rag(q)

In [ ]:
# FPAR comparison across all strategies
def stacked_rag_silent(q):
    return stacked_rag(q, verbose=False)

print('Running full FPAR comparison...')
fpar_stacked = run_fpar_eval(stacked_rag_silent, OOS_EVAL_SET, 'All three layers stacked')

print('\n' + '=' * 60)
print('FPAR Summary')
print('=' * 60)
strategies = [
    ('No guard (soft prompt)',    fpar_unguarded),
    ('Defense Layer 3 only',      fpar_guarded),
    ('All three layers stacked',  fpar_stacked),
]
for name, fpar in strategies:
    bar = '█' * int(fpar * 30)
    tag = '⚠️' if fpar > 0.05 else '✅'
    print(f'  {name:<30} {fpar:.0%}  {bar} {tag}')
print(f'\n  Target: < 5%')

In [ ]:
# Visualise: which layer catches which type of OOS question
layer_stats = {'Layer 3': [], 'Layer 2': [], 'Layer 1': [], 'Answered': []}

for item in OOS_EVAL_SET:
    r = stacked_rag(item['q'], verbose=False)
    if not r['abstained']:
        layer_stats['Answered'].append(item['category'])
    elif 'Layer 3' in r['layer']:
        layer_stats['Layer 3'].append(item['category'])
    elif 'Layer 2' in r['layer']:
        layer_stats['Layer 2'].append(item['category'])
    else:
        layer_stats['Layer 1'].append(item['category'])

layer_counts = {k: len(v) for k, v in layer_stats.items()}

fig, ax = plt.subplots(figsize=(9, 5))
colors = {'Layer 3': '#1565C0', 'Layer 2': '#6A1B9A', 'Layer 1': '#2E7D32', 'Answered': '#E53935'}
bars = ax.bar(layer_counts.keys(), layer_counts.values(),
              color=[colors[k] for k in layer_counts], alpha=0.85)

for bar, (k, v) in zip(bars, layer_counts.items()):
    label = '✅' if k != 'Answered' else '⚠️'
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.1, f'{v} {label}',
            ha='center', fontsize=11)

ax.set_ylabel('OOS Questions Caught')
ax.set_title(f'Where Each Layer Catches OOS Questions\n(Total: {len(OOS_EVAL_SET)} OOS questions)',
             fontweight='bold')
ax.set_ylim(0, max(layer_counts.values()) + 2)

legend_handles = [
    mpatches.Patch(color='#1565C0', label='Layer 3: retrieval threshold (cheapest — no LLM)'),
    mpatches.Patch(color='#6A1B9A', label='Layer 2: confidence gate (1 LLM call)'),
    mpatches.Patch(color='#2E7D32', label='Layer 1: hard prompt instruction (2 LLM calls)'),
    mpatches.Patch(color='#E53935', label='Answered (fabrication — should be 0)'),
]
ax.legend(handles=legend_handles, fontsize=9, loc='upper right')
show_plot()

---
## 8. Using a Real LLM (Claude API)

Replace the `MockLLM` with a real Claude API call to see the concepts in action
on a production-grade model.

Install: `pip install anthropic`

In [ ]:
# Set your API key: export ANTHROPIC_API_KEY=sk-ant-...
# or set it directly below (never commit keys to source control).

ANTHROPIC_AVAILABLE = False
try:
    import anthropic
    ANTHROPIC_AVAILABLE = bool(os.environ.get('ANTHROPIC_API_KEY'))
except ImportError:
    pass

if ANTHROPIC_AVAILABLE:
    client = anthropic.Anthropic()

    def claude_generate(prompt: str, model: str = 'claude-haiku-4-5-20251001') -> str:
        message = client.messages.create(
            model=model,
            max_tokens=512,
            messages=[{'role': 'user', 'content': prompt}],
        )
        return message.content[0].text

    def real_rag(query: str, verbose: bool = True) -> dict:
        docs, scores = retrieve(query, top_k=3, return_scores=True)
        if scores[0] < ABSTENTION_THRESHOLD:
            result = {'answer': ABSTENTION_PHRASE, 'abstained': True,
                      'reason': f'low_score={scores[0]:.3f}'}
        else:
            context = '\n\n'.join(f'[{d["title"]}]\n{d["text"]}' for d in docs)
            answer  = claude_generate(HARD_PROMPT.format(context=context, question=query))
            result  = {'answer': answer, 'abstained': is_abstention(answer),
                       'reason': f'score={scores[0]:.3f}'}
        if verbose:
            tag = '⛔ ABSTAINED' if result['abstained'] else '✅ ANSWERED'
            print(f'{tag} | {query}')
            print(f'        {result["answer"][:100]}\n')
        return result

    # Run a quick test
    test_qs = [
        'What is the maximum file upload size on the Pro plan?',
        'What is the warranty on the Quantum Teleporter?',
    ]
    print('=== Claude API (real LLM) ===')
    for q in test_qs:
        real_rag(q)

    print('\n=== FPAR with Claude ===')
    def real_rag_silent(q):
        return real_rag(q, verbose=False)
    run_fpar_eval(real_rag_silent, OOS_EVAL_SET, 'Claude with stacked defenses')

else:
    print('anthropic not installed or ANTHROPIC_API_KEY not set.')
    print()
    print('To enable this section:')
    print('  pip install anthropic')
    print('  export ANTHROPIC_API_KEY=sk-ant-...')
    print()
    print('The real_rag() function above shows the pattern you would use.')
    print('Replace mock_llm.generate() with claude_generate() anywhere in this notebook.')

---
## Key Takeaways

1. **LLMs are structurally biased toward fabrication.** RLHF trained them to prefer
   confident, complete answers. "I don't know" is literally what they were trained
   *not* to say. You are swimming upstream — the more explicit and rule-like your
   abstention instruction, the more likely the model complies.

2. **Soft prompts get soft compliance.** `"Answer based on the context"` is a suggestion,
   not a constraint. Switching from soft to hard instructions (`ONLY`, `EXACTLY`) can
   halve your FPAR with a one-line change.

3. **Classification before generation is more reliable than mid-sentence abstention.**
   Asking a YES/NO gate question changes the cognitive task. Models are much better
   at choosing from options than at refusing to generate.

4. **Retrieval-aware abstention is the cheapest defense.** A cosine similarity check
   costs microseconds. If your retriever found nothing useful, never call the LLM.

5. **Calibrate your threshold on labeled data.** Plot in-scope and OOS score
   distributions and pick the value that separates them. Typical sweet spot: 0.4–0.7,
   depending on embedding model and corpus.

6. **FPAR is the metric that matters for OOS questions.** Target < 5%. Include all
   four OOS categories in your eval set: adjacent topics, made-up entities,
   off-topic entirely, and plausible-but-absent.

7. **Don't over-abstain.** The goal is not to refuse everything uncertain.
   It is to refuse when context is insufficient, and answer confidently when it is.
   Monitor both FPAR (fabrications) and in-scope answer rate together.

---

*Up next: Lesson 7.5 — Evaluating hallucination at scale: faithfulness metrics and automated LLM judges.*